# COMPASS univariate models

Per-landmark univariate Cox arm, plus a nominal-significance filter over
those results. Requires `01_preprocessing.ipynb` to have built the merged
`profile_data` inputs under `prediction_inputs_<arm>/` first.

In [ ]:
ARMS = ["adt"]
ENDPOINTS = ("platinum", "nepc", "avpc", "avpc_nepc")
COHORTS = ("all", "metastatic", "localized")
OVERWRITE = False  # True: refit and replace existing outputs; False: resume/skip

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

cp.FORCE_RERUN = OVERWRITE
RUNS = cp.make_endpoint_runs(ARMS, endpoints=ENDPOINTS, cohorts=COHORTS)

## Run univariate models

Per-landmark univariate arm. Set `OVERWRITE = True` in the configuration cell to refit and replace existing outputs. With `False`, completed landmarks are skipped.

In [ ]:
for run in RUNS:
    cp.run_univariate(run)

## Nominally significant results

Filters the per-landmark univariate results to `p_value < 0.05` (nominal,
not multiplicity-adjusted -- `q_value` is retained in the export for that) and
writes per-run tables to `cox/nominally_significant_univariate_results.csv`
beneath the `profile_data` output root.

In [ ]:
NOMINAL_ALPHA = 0.05

nominal_tables = {}
for run in RUNS:
    results = cp.load_univariate_results(run)
    filtered = cp.filter_nominal(results, alpha=NOMINAL_ALPHA)
    nominal_tables[cp.run_key(run)] = filtered

    export_path = run["output_dir"] / "cox" / "nominally_significant_univariate_results.csv"
    if export_path.exists() and not OVERWRITE:
        print(f"{run['label']}: keeping existing nominal-significance export -> {export_path}")
    else:
        export_path.parent.mkdir(parents=True, exist_ok=True)
        filtered.to_csv(export_path, index=False)
        print(f"{run['label']}: {len(filtered)} nominally significant rows -> {export_path}")

In [ ]:
nominal_tables

## Supplemental PSA scale sensitivity analysis

Recomputes matched PSA summary features from the same underlying pre-landmark measurements on the raw and `log1p` scales, then fits the same age- and observation-count-adjusted univariate Cox models. Results are written separately beneath `cox_psa_scale_supplement/`; the primary univariate results are unchanged.

In [ ]:
psa_scale_tables = {}
for run in RUNS:
    psa_scale_tables[cp.run_key(run)] = cp.run_psa_scale_supplement(run)

In [ ]:
psa_scale_tables

## Separate sequencing, Gleason, and PRS univariate runs

Runs each indexed analysis for both endpoints, using the endpoint-specific `somatic_gleason/` inputs built by `01_preprocessing.ipynb`.

In [ ]:
for run in RUNS:
    cp.run_somatic_gleason_univariate(run)

In [ ]:
somatic_gleason_tables = {}
for run in RUNS:
    results = cp.load_somatic_gleason_univariate_results(run)
    filtered = cp.filter_nominal(results, alpha=NOMINAL_ALPHA)
    somatic_gleason_tables[cp.run_key(run)] = filtered
    export_path = run["output_dir"] / "cox_somatic_gleason" / "nominally_significant_univariate_results.csv"
    if export_path.exists() and not OVERWRITE:
        print(f"{run['label']}: keeping existing nominal sequencing/Gleason/PRS export -> {export_path}")
    else:
        export_path.parent.mkdir(parents=True, exist_ok=True)
        filtered.to_csv(export_path, index=False)
        print(f"{run['label']}: {len(filtered)} nominal sequencing/Gleason/PRS rows -> {export_path}")

In [ ]:
somatic_gleason_tables